In [ ]:
# Parameters - values are injected by Fabric Pipeline
layer = ""
entity = ""
pipeline_group = ""
orchestrator_run_id = ""

In [ ]:
import logging

from notebookutils import mssparkutils
from pyspark.sql import SparkSession

from tag_data_engineering.connectors.fabric_lakehouse_connector import FabricLakehouseConnector
from tag_data_engineering.extractors.blob_extractor import BlobExtractor
from tag_data_engineering.extractors.entra_users_extractor import EntraUsersExtractor
from tag_data_engineering.extractors.mysql_extractor import MySqlExtractor
from tag_data_engineering.extractors.postgres_extractor import PostgresExtractor
from tag_data_engineering.extractors.rest_api_extractor import RestApiExtractor
from tag_data_engineering.extractors.sql_server_extractor import SqlServerExtractor
from tag_data_engineering.extractors.verint_adherence_extractor import VerintAdherenceExtractor
from tag_data_engineering.pipeline.models import Layer
from tag_data_engineering.pipeline.pipeline_discoverer import PipelineDiscoverer
from tag_data_engineering.runners.lab_runner import LabRunner
from tag_data_engineering.runners.landing_copyjob_normalize_runner import LandingCopyJobNormalizeRunner
from tag_data_engineering.runners.landing_runner import LandingRunner
from tag_data_engineering.runners.setup_runner import SetupRunner
from tag_data_engineering.runners.transformation_runner import TransformationRunner
from tag_data_engineering.secrets.azure_secret_provider import AzureKeyVaultSecretProvider
from tag_data_engineering.secrets.fabric_notebook_credential import FabricNotebookCredential


logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

spark = SparkSession.builder.getOrCreate()
connector = FabricLakehouseConnector(spark, workspace_id="{{WORKSPACE_ID}}", sql_endpoint_id="{{SQL_ENDPOINT_ID}}")
secret_provider = AzureKeyVaultSecretProvider(vault_url="{{KEY_VAULT_URL}}", credential=FabricNotebookCredential())

logger.info(f"=== PARAMETERS: layer={repr(layer)}, entity={repr(entity)}, pipeline_group={repr(pipeline_group)}, orchestrator_run_id={repr(orchestrator_run_id)} ===")

if not layer:
    raise ValueError(f"layer parameter is empty! layer={repr(layer)}, entity={repr(entity)}")

metadata = None
if layer != "lab":
    discoverer = PipelineDiscoverer()
    # For copy job layers, discover from 'landing' and filter by sublayer
    if layer in ("landing_copyjob", "landing_copyjob_normalize"):
        discovered_jobs = [job for job in discoverer.discover_layer(Layer.LANDING) if job.layer.value == layer]
    else:
        discovered_jobs = discoverer.discover_layer(Layer(layer))

    if layer != "setup":
        matching_job = next((job for job in discovered_jobs if job.entity == entity), None)
        if not matching_job:
            available = [job.entity for job in discovered_jobs]
            raise ValueError(f"No job found for {layer}/{entity}. Available: {available}")
        metadata = matching_job.metadata

if layer == "setup":
    runner = SetupRunner(connector=connector)
    setup_output = runner.run(entity=entity, pipeline_group=pipeline_group, orchestrator_run_id=orchestrator_run_id)
    logger.info("✅ Setup complete")
    if setup_output:
        mssparkutils.notebook.exit(setup_output)
elif layer == "landing":
    runner = LandingRunner(
        connector=connector,
        extractors=[
            RestApiExtractor(secret_provider=secret_provider),
            EntraUsersExtractor(secret_provider=secret_provider),
            VerintAdherenceExtractor(secret_provider=secret_provider, connector=connector),
            BlobExtractor(secret_provider=secret_provider),
            MySqlExtractor(secret_provider=secret_provider),
            PostgresExtractor(secret_provider=secret_provider),
            SqlServerExtractor(secret_provider=secret_provider),
        ],
    )
    result = runner.run(metadata=metadata)
    logger.info(f"✅ Landing extraction: {result.records_extracted} records in {result.duration_seconds}s")
elif layer == "landing_copyjob":
    runner = LandingRunner(connector=connector, extractors=[])
    result = runner.run(metadata=metadata)
    logger.info(f"✅ Copy Job invoked: {result.records_extracted} records in {result.duration_seconds}s")
elif layer == "landing_copyjob_normalize":
    runner = LandingCopyJobNormalizeRunner(connector=connector, extractors=[])
    result = runner.run(metadata=metadata)
    logger.info(f"✅ Copy Job Normalized: {result.records_extracted} records in {result.duration_seconds}s")
elif layer in ("bronze", "silver", "gold"):
    runner = TransformationRunner(connector=connector)
    result = runner.run_transformation(metadata=metadata)
    logger.info(f"✅ Transformation: {result.rows_processed} rows in {result.duration_seconds}s")
elif layer == "lab":
    if not entity:
        raise ValueError("entity parameter is required for lab runs. Use entity='all' to execute all SQL files.")
    runner = LabRunner(connector=connector)
    applied = runner.run(layer=layer, entity=entity)
    logger.info(f"✅ Lab SQL applied: {len(applied)} file(s): {applied}")
else:
    raise ValueError(f"Unknown layer: {repr(layer)}")